# 🎯 Arabic ABSA System — Aspect-Based Sentiment Analysis
### Production-grade pipeline for Arabic customer reviews
**Architecture:** Two-model system using AraBERT  
- **Model 1:** Multi-label Aspect Detection (BCEWithLogitsLoss)  
- **Model 2:** Aspect-Conditioned Sentiment Classification (CrossEntropyLoss)


## 📦 1. Install Dependencies

In [ ]:
%%capture
!pip install transformers==4.40.0 datasets arabert torch scikit-learn pandas openpyxl tqdm

## 📚 2. Imports & Configuration

In [ ]:
import os, json, re, ast, warnings, random
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from transformers import (
    AutoTokenizer, AutoModel,
    get_linear_schedule_with_warmup
)
from sklearn.metrics import (
    f1_score, classification_report,
    confusion_matrix, multilabel_confusion_matrix
)
from sklearn.preprocessing import MultiLabelBinarizer
from tqdm import tqdm

warnings.filterwarnings('ignore')

# ── Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

# ── Paths (Kaggle style — adjust if needed)
DATA_DIR   = Path('/kaggle/input/your-dataset')   # ← change to your dataset path
OUTPUT_DIR = Path('/kaggle/working/models')
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Model
MODEL_NAME = 'aubmindlab/bert-base-arabertv02'
MAX_LEN    = 128

# ── Aspects
ASPECTS = ['food','service','price','cleanliness','delivery','ambiance','app_experience','general','none']
NUM_ASPECTS = len(ASPECTS)
ASPECT2ID = {a:i for i,a in enumerate(ASPECTS)}
ID2ASPECT  = {i:a for a,i in ASPECT2ID.items()}

# ── Sentiments
SENTIMENTS = ['positive', 'neutral', 'negative']
SENT2ID    = {s:i for i,s in enumerate(SENTIMENTS)}
ID2SENT    = {i:s for s,i in SENT2ID.items()}

print(f'Aspects  : {ASPECTS}')
print(f'Sentiments: {SENTIMENTS}')


## 🧹 3. Arabic Text Preprocessing Pipeline

In [ ]:
import unicodedata

# Emoji → Arabic sentiment words mapping
EMOJI_MAP = {
    '😊':'سعيد','😀':'سعيد','😁':'ممتاز','❤️':'رائع','💖':'رائع','👍':'جيد',
    '✅':'ممتاز','🌟':'ممتاز','⭐':'ممتاز','💯':'مثالي','🔥':'ممتاز',
    '😍':'رائع','🥰':'رائع','😢':'سيء','😡':'سيء','👎':'سيء',
    '💔':'سيء','😞':'سيء','😠':'سيء','🤬':'سيء','😤':'سيء',
    '😐':'عادي','🤔':'عادي','😑':'عادي','😶':'عادي',
}

def normalize_arabic(text: str) -> str:
    if not isinstance(text, str):
        return ''
    # Replace emojis with sentiment words
    for emoji, word in EMOJI_MAP.items():
        text = text.replace(emoji, f' {word} ')
    # Lowercase English chars
    text = text.lower()
    # Arabic letter normalization
    text = re.sub(r'[أإآ]', 'ا', text)
    text = re.sub(r'ى',     'ي', text)
    text = re.sub(r'ة',     'ه', text)
    text = re.sub(r'ؤ',     'و', text)
    text = re.sub(r'ئ',     'ي', text)
    # Remove diacritics (tashkeel)
    text = re.sub(r'[\u064B-\u065F\u0670]', '', text)
    # Remove elongation (tatweel)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)
    # Remove non-Arabic/English/numbers noise but keep spaces
    text = re.sub(r'[^\u0600-\u06FF\u0750-\u077F\uFB50-\uFDFF\uFE70-\uFEFF'
                  r'a-z0-9\s.,!?؟،؛:-]', ' ', text)
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Quick test
samples = [
    'الأكل ممتاز لكن الخدمة بطيئة والأسعار غالية ❤️',
    'التطبيق سيءءءء جدًا وما فيه خدمة عملاء 😡👎',
    'https://example.com مريييييح وهادئ 👍',
]
for s in samples:
    print(f'IN : {s}')
    print(f'OUT: {normalize_arabic(s)}')
    print()


## 📊 4. Load & Prepare Data

In [ ]:
def safe_parse_list(x):
    if isinstance(x, list): return x
    if isinstance(x, str):
        try: return json.loads(x)
        except:
            try: return ast.literal_eval(x)
            except: return []
    return []

def safe_parse_dict(x):
    if isinstance(x, dict): return x
    if isinstance(x, str):
        try: return json.loads(x)
        except:
            try: return ast.literal_eval(x)
            except: return {}
    return {}

def load_dataset(path):
    df = pd.read_excel(path)
    df['review_text'] = df['review_text'].astype(str)
    df['clean_text']  = df['review_text'].apply(normalize_arabic)
    if 'aspects' in df.columns:
        df['aspects_list']   = df['aspects'].apply(safe_parse_list)
        df['sentiments_dict']= df['aspect_sentiments'].apply(safe_parse_dict)
    return df

train_df = load_dataset(DATA_DIR / 'train_fixed.xlsx')
val_df   = load_dataset(DATA_DIR / 'validation_fixed.xlsx')
test_df  = load_dataset(DATA_DIR / 'unlabeled_fixed.xlsx')

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

# Aspect distribution in training
all_aspects = [a for row in train_df['aspects_list'] for a in row]
print('\nAspect distribution (train):')
for asp, cnt in Counter(all_aspects).most_common():
    bar = '█' * (cnt // 20)
    print(f'  {asp:<16} {cnt:>4}  {bar}')


## 🔵 5. Model 1 — Aspect Detection Dataset

In [ ]:
class AspectDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=MAX_LEN):
        self.texts    = df['clean_text'].tolist()
        self.tokenizer= tokenizer
        self.max_len  = max_len
        # Build multi-hot labels
        mlb = MultiLabelBinarizer(classes=ASPECTS)
        aspect_lists  = df['aspects_list'].tolist() if 'aspects_list' in df.columns else [[]]*len(df)
        self.labels   = mlb.fit_transform(aspect_lists).astype(np.float32)

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids'     : enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels'        : torch.tensor(self.labels[idx])
        }

print('AspectDataset class ready ✓')


## 🏗️ 6. Model 1 Architecture — Multi-label Aspect Detector

In [ ]:
class AspectDetector(nn.Module):
    def __init__(self, model_name, num_aspects, dropout=0.3):
        super().__init__()
        self.bert    = AutoModel.from_pretrained(model_name)
        hidden       = self.bert.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, num_aspects)
        )

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # Mean pooling over non-padding tokens
        mask_expanded = attention_mask.unsqueeze(-1).float()
        pooled = (out.last_hidden_state * mask_expanded).sum(1) / mask_expanded.sum(1)
        return self.classifier(self.dropout(pooled))

print('AspectDetector class ready ✓')


## 🔵 7. Model 2 — Sentiment Dataset

In [ ]:
class SentimentDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=MAX_LEN):
        self.pairs  = []
        self.labels = []
        self.tokenizer = tokenizer
        self.max_len   = max_len

        for _, row in df.iterrows():
            text  = row['clean_text']
            sents = row.get('sentiments_dict', {})
            if isinstance(sents, dict):
                for aspect, sentiment in sents.items():
                    if aspect in ASPECT2ID and sentiment in SENT2ID:
                        # Format: [ASPECT] <aspect> [TEXT] <review>
                        prompt = f'[ASPECT] {aspect} [TEXT] {text}'
                        self.pairs.append(prompt)
                        self.labels.append(SENT2ID[sentiment])

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.pairs[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids'     : enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels'        : torch.tensor(self.labels[idx], dtype=torch.long)
        }

print('SentimentDataset class ready ✓')


## 🏗️ 8. Model 2 Architecture — Sentiment Classifier

In [ ]:
class SentimentClassifier(nn.Module):
    def __init__(self, model_name, num_classes=3, dropout=0.3):
        super().__init__()
        self.bert    = AutoModel.from_pretrained(model_name)
        hidden       = self.bert.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # [CLS] token representation
        cls_out = out.last_hidden_state[:, 0, :]
        return self.classifier(self.dropout(cls_out))

print('SentimentClassifier class ready ✓')


## ⚙️ 9. Training Utilities

In [ ]:
def compute_class_weights(labels_flat, num_classes):
    """Inverse-frequency class weights for imbalanced data."""
    counts = np.bincount(labels_flat, minlength=num_classes)
    counts = np.where(counts == 0, 1, counts)
    weights = 1.0 / counts
    weights = weights / weights.sum() * num_classes
    return torch.tensor(weights, dtype=torch.float).to(DEVICE)

def train_aspect_model(model, train_loader, val_loader, epochs=4, lr=2e-5):
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(train_loader) * epochs
    scheduler   = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=total_steps//10, num_training_steps=total_steps
    )
    criterion = nn.BCEWithLogitsLoss()
    best_f1, best_state = 0.0, None

    for epoch in range(epochs):
        # ── Train
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs} [Aspect Train]'):
            input_ids = batch['input_ids'].to(DEVICE)
            attn_mask = batch['attention_mask'].to(DEVICE)
            labels    = batch['labels'].to(DEVICE)

            optimizer.zero_grad()
            logits = model(input_ids, attn_mask)
            loss   = criterion(logits, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)

        # ── Validate
        macro_f1, per_asp = eval_aspect_model(model, val_loader)
        print(f'  Epoch {epoch+1} | Loss: {avg_loss:.4f} | Macro F1: {macro_f1:.4f}')
        for asp, f1 in per_asp.items():
            print(f'    {asp:<16} F1={f1:.3f}')

        if macro_f1 > best_f1:
            best_f1    = macro_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            print(f'  ✓ New best saved (F1={best_f1:.4f})')

    model.load_state_dict(best_state)
    return model, best_f1

def eval_aspect_model(model, loader, thresholds=None):
    """Evaluate aspect detection with optional per-aspect thresholds."""
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attn_mask = batch['attention_mask'].to(DEVICE)
            logits    = model(input_ids, attn_mask).cpu().numpy()
            all_logits.append(logits)
            all_labels.append(batch['labels'].numpy())

    all_logits = np.concatenate(all_logits)
    all_labels = np.concatenate(all_labels)
    probs      = 1 / (1 + np.exp(-all_logits))  # sigmoid

    if thresholds is None:
        thresholds = [0.5] * NUM_ASPECTS
    preds = (probs > np.array(thresholds)).astype(int)

    macro_f1 = f1_score(all_labels, preds, average='macro', zero_division=0)
    per_asp  = {}
    for i, asp in enumerate(ASPECTS):
        per_asp[asp] = f1_score(all_labels[:, i], preds[:, i], zero_division=0)
    return macro_f1, per_asp

def tune_thresholds(model, loader):
    """Per-aspect threshold tuning over [0.2, 0.7]."""
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attn_mask = batch['attention_mask'].to(DEVICE)
            logits    = model(input_ids, attn_mask).cpu().numpy()
            all_logits.append(logits)
            all_labels.append(batch['labels'].numpy())
    all_logits = np.concatenate(all_logits)
    all_labels = np.concatenate(all_labels)
    probs      = 1 / (1 + np.exp(-all_logits))

    best_thresholds = []
    for i, asp in enumerate(ASPECTS):
        best_t, best_f = 0.5, 0.0
        for t in np.arange(0.2, 0.71, 0.05):
            preds = (probs[:, i] > t).astype(int)
            f1    = f1_score(all_labels[:, i], preds, zero_division=0)
            if f1 > best_f:
                best_f, best_t = f1, t
        best_thresholds.append(round(float(best_t), 2))
        print(f'  {asp:<16} best_threshold={best_t:.2f}  F1={best_f:.3f}')
    return best_thresholds

print('Training utilities ready ✓')


In [ ]:
def train_sentiment_model(model, train_loader, val_loader, class_weights, epochs=4, lr=2e-5):
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(train_loader) * epochs
    scheduler   = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=total_steps//10, num_training_steps=total_steps
    )
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    best_f1, best_state = 0.0, None

    for epoch in range(epochs):
        # ── Train
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs} [Sent Train]'):
            input_ids = batch['input_ids'].to(DEVICE)
            attn_mask = batch['attention_mask'].to(DEVICE)
            labels    = batch['labels'].to(DEVICE)

            optimizer.zero_grad()
            logits = model(input_ids, attn_mask)
            loss   = criterion(logits, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)

        # ── Validate
        macro_f1, report = eval_sentiment_model(model, val_loader)
        print(f'  Epoch {epoch+1} | Loss: {avg_loss:.4f} | Macro F1: {macro_f1:.4f}')
        print(report)

        if macro_f1 > best_f1:
            best_f1    = macro_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            print(f'  ✓ New best saved (F1={best_f1:.4f})')

    model.load_state_dict(best_state)
    return model, best_f1

def eval_sentiment_model(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attn_mask = batch['attention_mask'].to(DEVICE)
            logits    = model(input_ids, attn_mask)
            preds     = torch.argmax(logits, dim=-1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(batch['labels'].numpy())
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    report   = classification_report(all_labels, all_preds,
                    target_names=SENTIMENTS, zero_division=0)
    return macro_f1, report

print('Sentiment training utilities ready ✓')


## 🚀 10. Train Both Models

In [ ]:
# ── Load tokenizer once
print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ─────────────────────────────────────────────────
# MODEL 1: Aspect Detection
# ─────────────────────────────────────────────────
print('\n' + '='*60)
print('MODEL 1: Aspect Detection')
print('='*60)

train_asp_ds = AspectDataset(train_df, tokenizer)
val_asp_ds   = AspectDataset(val_df,   tokenizer)

train_asp_loader = DataLoader(train_asp_ds, batch_size=16, shuffle=True,  num_workers=2, pin_memory=True)
val_asp_loader   = DataLoader(val_asp_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

aspect_model = AspectDetector(MODEL_NAME, NUM_ASPECTS).to(DEVICE)
aspect_model, best_asp_f1 = train_aspect_model(
    aspect_model, train_asp_loader, val_asp_loader, epochs=4, lr=2e-5
)

# Threshold tuning
print('\n📐 Tuning per-aspect thresholds...')
BEST_THRESHOLDS = tune_thresholds(aspect_model, val_asp_loader)
print(f'\nFinal thresholds: {dict(zip(ASPECTS, BEST_THRESHOLDS))}')

# Save Model 1
asp_save_path = OUTPUT_DIR / 'aspect_model'
asp_save_path.mkdir(exist_ok=True)
torch.save(aspect_model.state_dict(), asp_save_path / 'model.pt')
tokenizer.save_pretrained(asp_save_path)
json.dump(BEST_THRESHOLDS, open(asp_save_path / 'thresholds.json', 'w'))
json.dump(ASPECTS, open(asp_save_path / 'aspect_labels.json', 'w'))
print(f'\n✓ Aspect model saved → {asp_save_path}')

# ─────────────────────────────────────────────────
# MODEL 2: Sentiment Classification
# ─────────────────────────────────────────────────
print('\n' + '='*60)
print('MODEL 2: Sentiment Classification')
print('='*60)

train_sent_ds = SentimentDataset(train_df, tokenizer)
val_sent_ds   = SentimentDataset(val_df,   tokenizer)

print(f'Sentiment pairs — Train: {len(train_sent_ds)} | Val: {len(val_sent_ds)}')
label_dist = Counter(train_sent_ds.labels)
print(f'Class distribution: { {ID2SENT[k]:v for k,v in label_dist.items()} }')

class_weights = compute_class_weights(train_sent_ds.labels, num_classes=3)
print(f'Class weights: {dict(zip(SENTIMENTS, class_weights.cpu().numpy().round(3)))}')

train_sent_loader = DataLoader(train_sent_ds, batch_size=16, shuffle=True,  num_workers=2, pin_memory=True)
val_sent_loader   = DataLoader(val_sent_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

sentiment_model = SentimentClassifier(MODEL_NAME, num_classes=3).to(DEVICE)
sentiment_model, best_sent_f1 = train_sentiment_model(
    sentiment_model, train_sent_loader, val_sent_loader,
    class_weights, epochs=4, lr=2e-5
)

# Save Model 2
sent_save_path = OUTPUT_DIR / 'sentiment_model'
sent_save_path.mkdir(exist_ok=True)
torch.save(sentiment_model.state_dict(), sent_save_path / 'model.pt')
tokenizer.save_pretrained(sent_save_path)
json.dump(SENTIMENTS, open(sent_save_path / 'sentiment_labels.json', 'w'))
print(f'\n✓ Sentiment model saved → {sent_save_path}')

print(f'\n🏆 Best Aspect F1   : {best_asp_f1:.4f}')
print(f'🏆 Best Sentiment F1: {best_sent_f1:.4f}')


## 📊 11. Detailed Evaluation

In [ ]:
print('='*60)
print('EVALUATION REPORT')
print('='*60)

# ── Aspect Model Evaluation
print('\n🔵 Aspect Detection (Validation):')
macro_f1, per_asp = eval_aspect_model(aspect_model, val_asp_loader, BEST_THRESHOLDS)
print(f'  Macro F1: {macro_f1:.4f}')
print(f'  {"Aspect":<16} {"F1":>6}')
print(f'  {"-"*24}')
for asp, f1 in sorted(per_asp.items(), key=lambda x:-x[1]):
    bar = '█' * int(f1 * 20)
    print(f'  {asp:<16} {f1:>5.3f}  {bar}')

# ── Sentiment Model Evaluation
print('\n🔵 Sentiment Classification (Validation):')
sent_macro_f1, report = eval_sentiment_model(sentiment_model, val_sent_loader)
print(f'  Macro F1: {sent_macro_f1:.4f}')
print(report)

# ── Confusion matrix
sentiment_model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in val_sent_loader:
        logits = sentiment_model(batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE))
        all_preds.extend(torch.argmax(logits, dim=-1).cpu().numpy())
        all_labels.extend(batch['labels'].numpy())

cm = confusion_matrix(all_labels, all_preds)
print('\nConfusion Matrix (rows=true, cols=pred):')
print(f'  {"":>10}', end='')
for s in SENTIMENTS: print(f'  {s:>10}', end='')
print()
for i, s in enumerate(SENTIMENTS):
    print(f'  {s:>10}', end='')
    for j in range(len(SENTIMENTS)):
        print(f'  {cm[i,j]:>10}', end='')
    print()


In [ ]:
class ABSAPredictor:
    """End-to-end inference pipeline for Arabic ABSA."""

    def __init__(self, aspect_model, sentiment_model, tokenizer,
                 thresholds, max_len=MAX_LEN):
        self.asp_model  = aspect_model.eval().to(DEVICE)
        self.sent_model = sentiment_model.eval().to(DEVICE)
        self.tokenizer  = tokenizer
        self.thresholds = np.array(thresholds)
        self.max_len    = max_len

    def _encode(self, text):
        return self.tokenizer(
            text, max_length=self.max_len,
            padding='max_length', truncation=True, return_tensors='pt'
        )

    @torch.no_grad()
    def predict_aspects(self, clean_text):
        enc   = self._encode(clean_text)
        inp   = enc['input_ids'].to(DEVICE)
        mask  = enc['attention_mask'].to(DEVICE)
        logit = self.asp_model(inp, mask).cpu().numpy()[0]
        probs = 1 / (1 + np.exp(-logit))
        detected = [ASPECTS[i] for i, p in enumerate(probs) if p > self.thresholds[i]]
        if not detected:
            # Fallback: pick top-1 if nothing passes threshold
            detected = [ASPECTS[int(np.argmax(probs))]]
        return detected

    @torch.no_grad()
    def predict_sentiment(self, clean_text, aspect):
        prompt = f'[ASPECT] {aspect} [TEXT] {clean_text}'
        enc    = self._encode(prompt)
        inp    = enc['input_ids'].to(DEVICE)
        mask   = enc['attention_mask'].to(DEVICE)
        logit  = self.sent_model(inp, mask)
        pred   = torch.argmax(logit, dim=-1).item()
        return ID2SENT[pred]

    def predict(self, review_id, raw_text):
        clean = normalize_arabic(raw_text)
        aspects = self.predict_aspects(clean)
        sentiments = {}
        for asp in aspects:
            sentiments[asp] = self.predict_sentiment(clean, asp)
        return {
            'review_id'        : int(review_id),
            'aspects'          : aspects,
            'aspect_sentiments': sentiments
        }

# Instantiate predictor
predictor = ABSAPredictor(aspect_model, sentiment_model, tokenizer, BEST_THRESHOLDS)

# ── Quick demo
demo_reviews = [
    (9999, 'الأكل ممتاز لكن الخدمة بطيئة والأسعار غالية'),
    (9998, 'التطبيق سريع ومريح والتوصيل في الموعد'),
    (9997, 'المكان نضيف والجو جميل لكن الاكل عادي'),
    (9996, 'عموما التجربة كانت مريحة'),
]
print('Demo Predictions:')
print('='*60)
for rid, text in demo_reviews:
    result = predictor.predict(rid, text)
    print(f'Review : {text}')
    print(f'Result : {json.dumps(result, ensure_ascii=False, indent=2)}')
    print()


In [ ]:
print('Running inference on unlabeled test set...')
results = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='Inferencing'):
    result = predictor.predict(row['review_id'], row['review_text'])
    results.append(result)

# Save JSONL submission
submission_path = OUTPUT_DIR / 'submission.jsonl'
with open(submission_path, 'w', encoding='utf-8') as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

# Also save as JSON array
json_path = OUTPUT_DIR / 'submission.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f'✓ Saved {len(results)} predictions')
print(f'  JSONL → {submission_path}')
print(f'  JSON  → {json_path}')

# Preview
print('\nSample predictions:')
for r in results[:3]:
    print(json.dumps(r, ensure_ascii=False, indent=2))
    print()


In [ ]:
# Aggregate stats over the submission
aspect_counts = Counter()
sentiment_counts = Counter()
multi_aspect = 0

for r in results:
    for a in r['aspects']:
        aspect_counts[a] += 1
    for s in r['aspect_sentiments'].values():
        sentiment_counts[s] += 1
    if len(r['aspects']) > 1:
        multi_aspect += 1

print(f'Total predictions : {len(results)}')
print(f'Multi-aspect reviews: {multi_aspect} ({multi_aspect/len(results)*100:.1f}%)')
print()
print('Aspect distribution in predictions:')
for a, c in aspect_counts.most_common():
    bar = '█' * (c // 50)
    print(f'  {a:<16} {c:>5}  {bar}')
print()
print('Sentiment distribution:')
for s, c in sentiment_counts.most_common():
    bar = '█' * (c // 50)
    print(f'  {s:<12} {c:>5}  {bar}')


## 💾 15. Save Complete Pipeline

In [ ]:
import pickle

# Save all pipeline metadata
pipeline_meta = {
    'aspects'        : ASPECTS,
    'sentiments'     : SENTIMENTS,
    'thresholds'     : BEST_THRESHOLDS,
    'model_name'     : MODEL_NAME,
    'max_len'        : MAX_LEN,
    'aspect2id'      : ASPECT2ID,
    'sent2id'        : SENT2ID,
    'best_aspect_f1' : best_asp_f1,
    'best_sent_f1'   : best_sent_f1,
}
json.dump(pipeline_meta, open(OUTPUT_DIR / 'pipeline_meta.json', 'w'), ensure_ascii=False, indent=2)

print('Files saved:')
for f in sorted(OUTPUT_DIR.rglob('*')):
    if f.is_file():
        size = f.stat().st_size / 1024
        print(f'  {f.relative_to(OUTPUT_DIR)}  ({size:.0f} KB)')

print('\n✅ Pipeline complete!')
print(f'   Aspect Detection  F1 : {best_asp_f1:.4f}')
print(f'   Sentiment Classif F1 : {best_sent_f1:.4f}')
